<!-- cspell:ignore colorbar dalitz fromkeys linalg mystnb yscale -->

# Phase space generators

:::{autolink-concat}
:::

Amplitude analysis needs samples of four-momenta that are distributed uniformly over the phase space of a decay. {class}`.TFPhaseSpaceGenerator` produces such samples with [`phasespace`](https://phasespace.readthedocs.io), which can run on TensorFlow, NumPy, or JAX. `phasespace` generates *weighted* events, so {class}`.TFPhaseSpaceGenerator` turns them into unweighted events with a hit-and-miss step: an event is kept if a uniform random number is below its weight. {class}`.TFWeightedPhaseSpaceGenerator` returns the weighted events instead. This page shows how to select a backend and how the `adaptive_max_weight` option makes the hit-and-miss step faster without changing the generated distribution.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
set_matplotlib_formats("svg")

## Backends

The backend is selected with the `backend` argument of the generator. All backends return a {obj}`.DataSample` of NumPy arrays with the same structure, so they can be swapped without changing the rest of an analysis.

In [ ]:
from tensorwaves.data import (
    NumpyUniformRNG,
    TFPhaseSpaceGenerator,
    TFUniformRealNumberGenerator,
)

initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}
generators = {
    "numpy": (
        TFPhaseSpaceGenerator(initial_state_mass, final_state_masses, backend="numpy"),
        NumpyUniformRNG(seed=0),
    ),
    "tensorflow": (
        TFPhaseSpaceGenerator(
            initial_state_mass, final_state_masses, backend="tensorflow"
        ),
        TFUniformRealNumberGenerator(seed=0),
    ),
    "jax": (
        TFPhaseSpaceGenerator(initial_state_mass, final_state_masses, backend="jax"),
        NumpyUniformRNG(seed=0),
    ),
}
for generator, _ in generators.values():
    generator.show_progress = False

| `backend` | Extra | Use of the {class}`.RealNumberGenerator` |
| --- | --- | --- |
| `"tensorflow"` (default) | `phsp` | {class}`.TFUniformRealNumberGenerator` is used directly, any other type only provides a seed |
| `"numpy"` | `phsp` | {class}`.NumpyUniformRNG` is used directly, any other type only provides a seed |
| `"jax"` | `phsp,jax` | any type only provides a seed |

The NumPy and JAX backends require `phasespace` 2.0 or higher. The installation of these extras is described on {doc}`/install`.

Every backend is used through the same {meth}`~.TFPhaseSpaceGenerator.generate` method. The events of different backends are not identical, not even with the same seed, but they follow the same distribution: a uniform phase space sample shows up as a flat Dalitz plot within the kinematic boundary.

In [ ]:
def invariant_mass_squared(*momenta: np.ndarray) -> np.ndarray:
    p = np.sum(momenta, axis=0)
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


backend_samples = {
    name: generator.generate(100_000, rng)
    for name, (generator, rng) in generators.items()
}
fig, axes = plt.subplots(
    ncols=len(backend_samples), figsize=(4 * len(backend_samples), 3.5), sharey=True
)
for ax, (name, sample) in zip(axes, backend_samples.items(), strict=True):
    s01 = invariant_mass_squared(sample["p0"], sample["p1"])
    s12 = invariant_mass_squared(sample["p1"], sample["p2"])
    ax.hist2d(s01, s12, bins=60, cmin=1, rasterized=True)
    ax.set_title(name)
    ax.set_xlabel(R"$m_{01}^2$ [GeV$^2$]")
axes[0].set_ylabel(R"$m_{12}^2$ [GeV$^2$]")
fig.tight_layout()
plt.show()

TensorFlow is the default, because it reproduces the samples of earlier TensorWaves versions, when combined with a {class}`.TFUniformRealNumberGenerator`. NumPy needs no compilation, so it is the fastest choice for a few small samples. TensorFlow and JAX compile functions on the first call, but are several times faster afterwards, so they are worth it when generating large or many samples for the same decay. JAX can be useful if the rest of the analysis already uses it. {doc}`TR-039 <compwa-report:039/index>` benchmarks the backends and compares their distributions for three-body and four-body decays.

There are two things to keep in mind:

- The same seed gives a different sample on each backend. Samples are only reproducible within one backend.
- `phasespace` selects its backend per process, and a generator activates its backend each time it generates. Each switch makes `phasespace` recompile, so alternating between backends is slow, and generators with different backends should not be used concurrently in threads.

## Adaptive maximum weight

By default, the random numbers of the hit-and-miss step are drawn up to $1$, the upper bound of the normalized weights that `phasespace` guarantees. The weights that actually occur are often far below that bound, especially for decays into many particles, so most events are rejected. With `adaptive_max_weight=True`, the random numbers are drawn up to an estimate of the maximum weight instead. The estimate is the largest weight seen so far plus a margin of 5%. If a later bunch contains a larger weight, the estimate is raised and the sample is generated anew, so that the result remains unbiased.

The option is keyword-only and is off by default, so existing code keeps producing the same events for the same seed.

The following plots illustrate this for a $5\,\mathrm{GeV}$ particle that decays into an increasing number of pions. The left panel shows the weight distributions relative to the largest weight in each sample. The right panel shows the fraction of events that survive the hit-and-miss step: with the default setting, this is the mean weight, while with `adaptive_max_weight=True` it is the mean weight divided by the estimated maximum.

In [ ]:
from tensorwaves.data import TFWeightedPhaseSpaceGenerator

n_particles = list(range(3, 9))
weights = {}
for n in n_particles:
    weighted_generator = TFWeightedPhaseSpaceGenerator(
        initial_state_mass=5.0,
        final_state_masses=dict.fromkeys(range(n), 0.13957),
        backend="numpy",
    )
    rng = NumpyUniformRNG(seed=0)
    weights[n] = weighted_generator.generate(100_000, rng)["weights"]

fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 3.8), layout="constrained")
for n, w in weights.items():
    ax1.hist(w / w.max(), bins=60, histtype="step", density=True, label=f"{n} pions")
ax1.set_xlabel("Weight / maximum weight in sample")
ax1.set_yticks([])
ax1.legend()
ax2.plot(n_particles, [w.mean() for w in weights.values()], "o-", label="Default")
ax2.plot(
    n_particles,
    [w.mean() / (1.05 * w.max()) for w in weights.values()],
    "s-",
    label="adaptive_max_weight=True",
)
ax2.set_xlabel("Number of pions")
ax2.set_ylabel("Fraction of accepted events")
ax2.set_yscale("log")
ax2.legend()
plt.show()

The shape of the weight distribution relative to its maximum changes only moderately with more particles, but the mean weight relative to the bound of $1$ drops by several orders of magnitude. With the estimated maximum, the fraction of accepted events stays above 10%. The gain in speed grows accordingly: for five or more pions, generation is orders of magnitude faster. {doc}`TR-038 <compwa-report:038/index>` benchmarks this and confirms with statistical tests that both settings produce the same distributions.

### Usage

The option is a keyword argument of {class}`.TFPhaseSpaceGenerator`. The following example generates a sample of a three-body decay with both settings. The events differ, because the hit-and-miss step consumes the random numbers differently, but both samples are distributed uniformly over phase space: the events fill the kinematically allowed region of the Dalitz plot evenly.

In [ ]:
initial_state_mass = 3.0969
final_state_masses = {0: 0.1350, 1: 0.1396, 2: 0.4937}
default_generator = TFPhaseSpaceGenerator(initial_state_mass, final_state_masses)
adaptive_generator = TFPhaseSpaceGenerator(
    initial_state_mass, final_state_masses, adaptive_max_weight=True
)

In [ ]:
size = 500_000
default_generator.show_progress = False
adaptive_generator.show_progress = False
samples = {
    "Default": default_generator.generate(size, TFUniformRealNumberGenerator(seed=0)),
    "adaptive_max_weight=True": adaptive_generator.generate(
        size, TFUniformRealNumberGenerator(seed=0)
    ),
}

In [ ]:
variables = {
    label: (
        invariant_mass_squared(sample["p0"], sample["p1"]),
        invariant_mass_squared(sample["p1"], sample["p2"]),
    )
    for label, sample in samples.items()
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True, layout="constrained")
for ax, (label, (m12, m23)) in zip(axes, variables.items(), strict=True):
    ax.hist2d(m12, m23, bins=150, cmin=1, rasterized=True)
    ax.set_title(label)
    ax.set_xlabel(R"$m^2_{12}$ [GeV$^2$]")
axes[0].set_ylabel(R"$m^2_{23}$ [GeV$^2$]")
fig.colorbar(axes[0].collections[0], ax=axes, label="Events")
plt.show()

### When to use it

The option pays off for decays into five or more particles, where the default setting rejects most events. Keep the default when samples have to be identical to those of earlier versions for the same seed, for instance when tests compare against stored values. A small `bunch_size` makes the estimate of the maximum weight less accurate and leads to more restarts, so the option works best with the default bunch size.